# 用 Deep Agents 编排受控的进度查询工具

本 notebook 继续使用仓库现有的 **8 份输变电工程进度 TXT**。核心问题不是“怎样让 Agent 自由搜索”，而是：在事实查询已经有可靠 Evidence Gate 后，什么时候值得让 LLM 规划多次只读查询？

先给结论边界：

- 单跳日期查询优先调用直接 reliable pipeline，不需要 Agent。
- 复合汇总可能受益于 planning + tool calling，但每个事实仍必须单独通过同一个 Evidence Gate。
- Agent 不直接接触 Chroma、原始文件、shell、`.env`、长期记忆或 subagent；它只能调用一个受限工具。

## 1. Deep Agents 的设计理念，以及它位于哪一层

**Deep Agents 是建立在 LangChain/LangGraph 上的 opinionated agent harness（带明确默认能力的 Agent 脚手架）**。LangGraph 让工程师显式画状态图；Deep Agents 则面向路径不完全预知的长任务，组合模型、工具、上下文管理、临时文件空间、skills、memory 和 subagents。

这些默认能力不是免费的：Deep Agents 依赖支持 tool calling 的 LLM，比直接 pipeline 更慢、更贵，也更难保证每次走相同路径。框架负责**编排**，不负责工程路由、记录匹配和日期正确性。

本知识库只有 8 个文档、541 条短任务记录：

- **filesystem**：适合长任务中管理大量中间产物；这里没有此需求，而且不应开放宿主文件。
- **skills/memory**：适合复用长期方法或偏好；这里的事实应来自版本化语料，不能由记忆替代。
- **subagents**：适合隔离大上下文或不同专业工具；当前结果集很小，多开只会增加成本和非确定性。
- **StateBackend**：保留框架的线程内状态能力，但不映射真实文件系统。

因此首版只保留受限的 planning + tool calling + StateBackend，并用实际运行决定 Agent 是否值得。

In [1]:
from __future__ import annotations

import json
import os
import re
import sys
import time
from importlib.metadata import version
from pathlib import Path
from typing import Any, Literal

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")


def find_repo_root(start: Path | None = None) -> Path:
    """从仓库根目录或 ZZworkbench 启动时都能定位资源。"""

    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        corpus = candidate / "knowledge" / "project_progress" / "texts"
        if corpus.is_dir() and (candidate / "ZZworkbench").is_dir():
            return candidate
    raise FileNotFoundError("Cannot locate the pipelines_rag repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from deepagents import (
    GeneralPurposeSubagentProfile,
    HarnessProfile,
    create_deep_agent,
    register_harness_profile,
)
from deepagents.backends import StateBackend
from dotenv import dotenv_values
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field, field_validator

from ZZworkbench.project_progress_reliable import (
    ProjectProgressKnowledgeBase,
    evaluate_reliable_lookup,
)
from ZZworkbench.rag_langchain.text_retrieval import (
    DEFAULT_CORPUS_ROOT,
    DEFAULT_INDEX_ROOT,
    ChunkConfig,
    EmbeddingConfig,
    RetrievalHit,
    build_embeddings,
    build_or_reuse_chroma,
    evaluate_retriever,
    load_eval_cases,
    load_txt_documents,
    open_chroma,
    retrieve_hybrid,
    split_documents,
)

print(
    {
        "repository_located": REPO_ROOT.name == "pipelines_rag",
        "langgraph": version("langgraph"),
        "deepagents": version("deepagents"),
        "langchain": version("langchain"),
    }
)

{'repository_located': True, 'langgraph': '1.2.11', 'deepagents': '0.7.8', 'langchain': '1.3.18'}


## 2. 同一语料、同一可靠合同

Deep Agent 不能拥有第二套事实规则。下面复用 `ProjectProgressKnowledgeBase`：查询规范化、实际工程别名、完整任务记录、父子层级、字段完整性、引用与 `exact/ambiguous/not_found/insufficient/conflict` 都与 LangGraph、直接 pipeline 相同。

hybrid 只负责候选召回；启用 retriever 时，如果精确记录没有真的出现在候选里，Evidence Gate 返回 `insufficient`，不会绕过检索后宣称 RAG 成功。

In [2]:
chunk_config = ChunkConfig()
documents = load_txt_documents(DEFAULT_CORPUS_ROOT, version="v4")
chunks = split_documents(documents, chunk_config)
knowledge_base = ProjectProgressKnowledgeBase(documents, chunks)

embedding_config = EmbeddingConfig()
embeddings = build_embeddings(embedding_config)
index_result = build_or_reuse_chroma(
    documents,
    chunks,
    embeddings,
    embedding_config,
    persist_directory=DEFAULT_INDEX_ROOT,
    corpus_root=DEFAULT_CORPUS_ROOT,
    version="v4",
    chunk_config=chunk_config,
)
vector_store = open_chroma(DEFAULT_INDEX_ROOT, embeddings)


def hybrid_candidates(query: str) -> list[RetrievalHit]:
    return retrieve_hybrid(
        vector_store,
        query,
        k=8,
        fetch_k=30,
        dense_weight=0.35,
    )


print(
    {
        "documents": len(documents),
        "chunks": len(chunks),
        "records": len(knowledge_base.records),
        "index_reused": index_result.reused,
    }
)

{'documents': 8, 'chunks': 63, 'records': 541, 'index_reused': True}


## 3. A/B/C 对照仍然成立

Agent 不替代检索评测。先在同一批 8 条正例上比较 dense-only 与 hybrid，再把原 8 条和新增 5 条可靠性样例一起交给 reliable pipeline。负例不适合用“是否召回某个答案词”衡量，而应看歧义与拒答状态。

In [3]:
eval_dir = REPO_ROOT / "knowledge" / "project_progress" / "evals"
positive_cases = load_eval_cases(eval_dir / "retrieval_v4.jsonl")
reliability_cases = load_eval_cases(eval_dir / "reliability_v4.jsonl")
all_cases = [*positive_cases, *reliability_cases]

dense_report = evaluate_retriever(
    vector_store, positive_cases, k=4, strategy="dense"
)
hybrid_report = evaluate_retriever(
    vector_store,
    positive_cases,
    k=4,
    strategy="hybrid",
    fetch_k=20,
    dense_weight=0.35,
)
reliable_report = evaluate_reliable_lookup(
    knowledge_base,
    all_cases,
    retriever=hybrid_candidates,
)
print(
    json.dumps(
        {
            "dense-only": {
                "source_hit@4": dense_report["source_hit_at_k"],
                "term_hit@4": dense_report["term_hit_at_k"],
            },
            "hybrid": {
                "source_hit@4": hybrid_report["source_hit_at_k"],
                "term_hit@4": hybrid_report["term_hit_at_k"],
            },
            "reliable": {
                "cases": reliable_report["cases"],
                "status_accuracy": reliable_report["status_accuracy"],
                "field_accuracy": reliable_report["field_accuracy"],
                "pass_rate": reliable_report["pass_rate"],
            },
        },
        ensure_ascii=False,
        indent=2,
    )
)

{
  "dense-only": {
    "source_hit@4": 0.5,
    "term_hit@4": 0.0
  },
  "hybrid": {
    "source_hit@4": 1.0,
    "term_hit@4": 1.0
  },
  "reliable": {
    "cases": 13,
    "status_accuracy": 1.0,
    "field_accuracy": 1.0,
    "pass_rate": 1.0
  }
}


## 4. 只暴露一个只读工具

工具输入使用 Pydantic schema；`fields` 只能是 `start_date/end_date/duration`，候选数量与检索策略封装在工具内部。Agent 无权传入任意 `k`、改索引或读取原始目录。

工具会原样返回 Evidence Gate 状态、完整记录、引用和诊断。`ambiguous/not_found/conflict` 不能由 Agent 改写成确定答案。工具输出中的原始证据只用于核对，不代表模型计划本身是证据。

In [4]:
AllowedField = Literal["start_date", "end_date", "duration"]


class ScheduleLookupInput(BaseModel):
    project: str = Field(
        min_length=1,
        description="工程名、知识库内简称；不知道工程时传“未指定工程”",
    )
    task: str = Field(min_length=1, description="要精确匹配的任务名称")
    fields: list[AllowedField] = Field(
        min_length=1,
        max_length=3,
        description="需要返回的任务记录字段",
    )

    @field_validator("fields")
    @classmethod
    def fields_must_be_unique(
        cls, fields: list[AllowedField]
    ) -> list[AllowedField]:
        if len(fields) != len(set(fields)):
            raise ValueError("fields cannot contain duplicates")
        return fields


@tool("lookup_project_schedule", args_schema=ScheduleLookupInput)
def lookup_project_schedule(
    project: str,
    task: str,
    fields: list[AllowedField],
) -> str:
    """只读查询一个工程任务；只有 status=exact 的记录可以作为答案事实。"""

    result = knowledge_base.lookup_fields(
        project=project,
        task=task,
        fields=fields,
        retriever=hybrid_candidates,
    )
    payload = result.to_dict()
    payload["citations"] = [
        {
            "source_name": record.source_name,
            "task_id": record.task_id,
            "chunk_id": record.chunk_id,
            "page": record.page,
        }
        for record in result.records
    ]
    payload["diagnostics"]["candidate_policy"] = {
        "k": 8,
        "fetch_k": 30,
        "dense_weight": 0.35,
        "fallbacks": 0,
    }
    return json.dumps(payload, ensure_ascii=False)


schema = lookup_project_schedule.args_schema.model_json_schema()
print(json.dumps(schema, ensure_ascii=False, indent=2))

{
  "properties": {
    "project": {
      "description": "工程名、知识库内简称；不知道工程时传“未指定工程”",
      "minLength": 1,
      "title": "Project",
      "type": "string"
    },
    "task": {
      "description": "要精确匹配的任务名称",
      "minLength": 1,
      "title": "Task",
      "type": "string"
    },
    "fields": {
      "description": "需要返回的任务记录字段",
      "items": {
        "enum": [
          "start_date",
          "end_date",
          "duration"
        ],
        "type": "string"
      },
      "maxItems": 3,
      "minItems": 1,
      "title": "Fields",
      "type": "array"
    }
  },
  "required": [
    "project",
    "task",
    "fields"
  ],
  "title": "ScheduleLookupInput",
  "type": "object"
}


In [5]:
boundary_examples = {
    "exact": {
        "project": "南溪输变电工程",
        "task": "地基基础施工",
        "fields": ["start_date", "end_date", "duration"],
    },
    "ambiguous": {
        "project": "未指定工程",
        "task": "施工准备",
        "fields": ["end_date"],
    },
    "not_found": {
        "project": "南溪输变电工程",
        "task": "锅炉点火",
        "fields": ["end_date"],
    },
}
for label, arguments in boundary_examples.items():
    payload = json.loads(lookup_project_schedule.invoke(arguments))
    print(label, payload["status"], payload["answer"].splitlines()[0])

exact exact 珠海110千伏南溪（旅游）输变电工程施工进度计划横道图中，“地基基础施工”计划开始2025年8月2日，计划完成2025年10月22日，工期82工作日。
ambiguous ambiguous 任务“施工准备”出现在多个工程文档中，请补充工程范围：110kV黄金输变电工程三级进度计划.txt、三级进度计划-土建.txt、三虎输变电工程三级进度计划土建部分.txt、珠海110千伏江湾输变电工程施工进度计划（202.txt


not_found not_found 知识库中没有找到任务“锅炉点火”的可验证记录。


## 5. 用 HarnessProfile 收回默认能力

Deep Agents 0.7.8 的 `tools=` 是**追加**工具，并不会自动移除内置工具。因此这里显式注册 `HarnessProfile`：

- 排除 `ls/read_file/write_file/edit_file/delete/glob/grep/execute`；
- 禁用自动添加的 general-purpose subagent，从而不暴露 `task`；
- 不传 skills、memory、宿主文件 backend；
- 使用 `StateBackend()`，只保留 graph state 内的临时后端；
- 用 `ToolCallLimitMiddleware` 把一次运行的查询工具调用限制为最多 4 次。

这里使用仓库既有的 OpenAI-compatible 配置风格。只检查 key 是否存在，不打印 key 或 `.env` 内容；还必须显式设置 `RUN_DEEPAGENTS_NETWORK=1` 才会发送问题与证据。否则仍构造完整 Agent，并跳过网络调用。

In [6]:
env_file = dotenv_values(REPO_ROOT / ".env") if (REPO_ROOT / ".env").is_file() else {}


def configured_value(*names: str, default: str | None = None) -> str | None:
    for name in names:
        if value := os.getenv(name):
            return value
        if value := env_file.get(name):
            return str(value)
    return default


api_key = configured_value("DEEPSEEK_API_KEY", "deepseek_api_key")
model_name = configured_value(
    "DEEPSEEK_MODEL", "deepseek_model", default="deepseek-chat"
)
network_opt_in = os.getenv("RUN_DEEPAGENTS_NETWORK") == "1"
base_url = configured_value(
    "DEEPSEEK_BASE_URL",
    "deepseek_base_url",
    default="https://api.deepseek.com",
)
assert model_name is not None and base_url is not None

chat_model = ChatOpenAI(
    model=model_name,
    api_key=api_key or "EMPTY",
    base_url=base_url,
    temperature=0,
    timeout=120,
    max_retries=1,
)

excluded_builtins = frozenset(
    {
        "ls",
        "read_file",
        "write_file",
        "edit_file",
        "delete",
        "glob",
        "grep",
        "execute",
    }
)
restricted_profile = HarnessProfile(
    excluded_tools=excluded_builtins,
    general_purpose_subagent=GeneralPurposeSubagentProfile(enabled=False),
)
register_harness_profile(f"openai:{model_name}", restricted_profile)

system_prompt = """
你只处理当前输变电工程进度知识库的只读查询。
先列出需要核验的独立任务，再对每个任务调用 lookup_project_schedule 一次。
只有工具返回 status=exact 时才能陈述该事实；其他状态必须原样标记。
不得补写工具没有返回的日期、工期或来源。最终答案逐项附 source_name、task_id、chunk_id。
单项失败不得影响其他已验证项。不要调用任何文件、shell、记忆或 subagent 能力。
""".strip()

agent_checkpointer = InMemorySaver()
agent = create_deep_agent(
    model=chat_model,
    tools=[lookup_project_schedule],
    system_prompt=system_prompt,
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="lookup_project_schedule",
            run_limit=4,
            exit_behavior="end",
        )
    ],
    subagents=[],
    skills=None,
    memory=None,
    backend=StateBackend(),
    checkpointer=agent_checkpointer,
    name="project_progress_compound_lookup",
)

print(
    {
        "agent_constructed": agent is not None,
        "network_model_available": bool(api_key),
        "network_run_enabled": bool(api_key) and network_opt_in,
        "excluded_builtin_count": len(restricted_profile.excluded_tools),
        "general_purpose_subagent_enabled": (
            restricted_profile.general_purpose_subagent.enabled
        ),
        "graph_nodes": sorted(agent.get_graph().nodes),
    }
)

{'agent_constructed': True, 'network_model_available': True, 'network_run_enabled': False, 'excluded_builtin_count': 8, 'general_purpose_subagent_enabled': False, 'graph_nodes': ['PatchToolCallsMiddleware.before_agent', 'ToolCallLimitMiddleware[lookup_project_schedule].after_model', '__end__', '__start__', 'model', 'tools']}


## 6. 单跳问题：负责任地证明“不需要 Agent”

对同一个问题先运行直接 reliable pipeline，再直接调用工具。两者不需要 LLM，应该给出相同记录和引用。随后若 key 可用，只做一次受限 Agent 运行，并记录 LLM/tool 次数、耗时和 token；若外部模型不可用则明确跳过，不影响工具合同验证。

In [7]:
single_question = "南溪输变电工程的地基基础施工在什么时间？"
single_arguments = {
    "project": "南溪输变电工程",
    "task": "地基基础施工",
    "fields": ["start_date", "end_date"],
}

direct_started = time.perf_counter()
direct_single = knowledge_base.lookup(
    single_question,
    retriever=hybrid_candidates,
)
direct_elapsed = time.perf_counter() - direct_started

tool_started = time.perf_counter()
tool_single = json.loads(lookup_project_schedule.invoke(single_arguments))
tool_elapsed = time.perf_counter() - tool_started

assert direct_single.status == tool_single["status"] == "exact"
assert direct_single.records[0].record_id == tool_single["records"][0]["record_id"]
print(
    {
        "status": tool_single["status"],
        "same_record": True,
        "same_source": (
            direct_single.records[0].source_name
            == tool_single["citations"][0]["source_name"]
        ),
        "direct_retrievals": direct_single.retrieval_calls,
        "tool_retrievals": tool_single["retrieval_calls"],
        "direct_seconds": round(direct_elapsed, 4),
        "tool_seconds": round(tool_elapsed, 4),
        "llm_calls": 0,
    }
)

{'status': 'exact', 'same_record': True, 'same_source': True, 'direct_retrievals': 1, 'tool_retrievals': 1, 'direct_seconds': 0.2672, 'tool_seconds': 0.1859, 'llm_calls': 0}


In [8]:
def message_text(message: Any) -> str:
    content = getattr(message, "content", "")
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(
            str(block.get("text", "")) if isinstance(block, dict) else str(block)
            for block in content
        )
    return str(content)


def invoke_agent_safely(question: str, thread_id: str) -> dict[str, Any]:
    if not api_key:
        return {
            "ran": False,
            "skip_reason": "DEEPSEEK_API_KEY is not configured",
        }
    if not network_opt_in:
        return {
            "ran": False,
            "skip_reason": "network run requires RUN_DEEPAGENTS_NETWORK=1",
        }

    started = time.perf_counter()
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": question}]},
            config={
                "configurable": {"thread_id": thread_id},
                "recursion_limit": 16,
            },
        )
    except Exception as exc:
        return {
            "ran": False,
            "skip_reason": f"model call unavailable: {type(exc).__name__}",
        }

    messages = result["messages"]
    calls = []
    tool_outputs = []
    llm_calls = 0
    input_tokens = 0
    output_tokens = 0
    for message in messages:
        if isinstance(message, AIMessage):
            llm_calls += 1
            calls.extend(message.tool_calls)
            usage = message.usage_metadata or {}
            input_tokens += int(usage.get("input_tokens", 0) or 0)
            output_tokens += int(usage.get("output_tokens", 0) or 0)
        elif isinstance(message, ToolMessage):
            try:
                tool_outputs.append(json.loads(message_text(message)))
            except json.JSONDecodeError:
                tool_outputs.append({"status": "invalid_tool_output"})

    return {
        "ran": True,
        "elapsed_seconds": time.perf_counter() - started,
        "llm_calls": llm_calls,
        "tool_calls": calls,
        "tool_outputs": tool_outputs,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "final_answer": message_text(messages[-1]),
    }


single_agent_run = invoke_agent_safely(
    single_question,
    "deepagents-single-hop",
)
if single_agent_run["ran"]:
    print(
        json.dumps(
            {
                "ran": True,
                "llm_calls": single_agent_run["llm_calls"],
                "tool_call_count": len(single_agent_run["tool_calls"]),
                "elapsed_seconds": round(single_agent_run["elapsed_seconds"], 3),
                "input_tokens": single_agent_run["input_tokens"],
                "output_tokens": single_agent_run["output_tokens"],
                "tool_inputs": [
                    call.get("args") for call in single_agent_run["tool_calls"]
                ],
                "final_answer": single_agent_run["final_answer"][:1200],
            },
            ensure_ascii=False,
            indent=2,
        )
    )
else:
    print(single_agent_run)

{'ran': False, 'skip_reason': 'network run requires RUN_DEEPAGENTS_NETWORK=1'}


## 7. 复合问题：Agent 才开始有合理用例

真实复合任务：

> 汇总南溪地基基础施工的计划起止时间、黄金主体结构封顶的计划完成时间、三虎电气全站电缆敷设安装的计划起止时间，并分别给出来源；任何一项证据不足都单独标记。

这需要拆成 3 次受控 lookup、分别保留状态与引用，再统一表达。一项失败不能污染另外两项。先运行确定性工具基线，确认 Agent 唯一允许看到的事实集合。

In [9]:
compound_question = (
    "汇总南溪地基基础施工的计划起止时间、黄金主体结构封顶的计划完成时间、"
    "三虎电气全站电缆敷设安装的计划起止时间，并分别给出来源；"
    "任何一项证据不足都单独标记。"
)
compound_arguments = [
    {
        "project": "南溪输变电工程",
        "task": "地基基础施工",
        "fields": ["start_date", "end_date"],
    },
    {
        "project": "黄金输变电工程",
        "task": "主体结构封顶",
        "fields": ["end_date"],
    },
    {
        "project": "三虎电气",
        "task": "全站电缆敷设安装",
        "fields": ["start_date", "end_date"],
    },
]

compound_baseline = [
    json.loads(lookup_project_schedule.invoke(arguments))
    for arguments in compound_arguments
]
print(
    json.dumps(
        [
            {
                "task": payload["records"][0]["task_name"],
                "status": payload["status"],
                "start_date": payload["records"][0]["start_date"],
                "end_date": payload["records"][0]["end_date"],
                "citation": payload["citations"][0],
            }
            for payload in compound_baseline
        ],
        ensure_ascii=False,
        indent=2,
    )
)
assert all(payload["status"] == "exact" for payload in compound_baseline)

[
  {
    "task": "地基基础施工",
    "status": "exact",
    "start_date": "2025年8月2日",
    "end_date": "2025年10月22日",
    "citation": {
      "source_name": "南溪三级进度.txt",
      "task_id": "1",
      "chunk_id": "chunk-90b947ba042692887b78b4cd",
      "page": 1
    }
  },
  {
    "task": "主体结构封顶",
    "status": "exact",
    "start_date": "2024年8月30日",
    "end_date": "2024年10月24日",
    "citation": {
      "source_name": "110kV黄金输变电工程三级进度计划.txt",
      "task_id": "10",
      "chunk_id": "chunk-b5c87ce1a6909e50abf0dd95",
      "page": 1
    }
  },
  {
    "task": "全站电缆敷设安装",
    "status": "exact",
    "start_date": "2023年4月30日",
    "end_date": "2023年6月2日",
    "citation": {
      "source_name": "三虎输变电工程三级进度计划电气部分.txt",
      "task_id": "3",
      "chunk_id": "chunk-4bb0d9954107e8571c8756ad",
      "page": 1
    }
  }
]


In [10]:
compound_agent_run = invoke_agent_safely(
    compound_question,
    "deepagents-compound",
)
if compound_agent_run["ran"]:
    trace = {
        "plan_and_tool_inputs": [
            {"name": call.get("name"), "args": call.get("args")}
            for call in compound_agent_run["tool_calls"]
        ],
        "structured_tool_returns": [
            {
                "status": payload.get("status"),
                "citations": payload.get("citations", []),
            }
            for payload in compound_agent_run["tool_outputs"]
        ],
        "final_answer": compound_agent_run["final_answer"][:1800],
    }
    print(json.dumps(trace, ensure_ascii=False, indent=2))
else:
    print(compound_agent_run)

{'ran': False, 'skip_reason': 'network run requires RUN_DEEPAGENTS_NETWORK=1'}


## 8. Agent 层评测

不能用“最终回答看起来不错”代替验证。这里检查工具选择、参数覆盖、最终答案中已验证日期和引用覆盖、额外日期声明，以及工具边界对歧义/拒答状态的保留。

`token` 与耗时只在真实网络调用成功时报告。不同模型与网络条件下它们会变化；这正是 Agent 相比直接 pipeline 的额外成本，而不是检索质量指标。

In [11]:
ZH_DATE = re.compile(r"(\d{4})年(\d{1,2})月(\d{1,2})日")
ISO_DATE = re.compile(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})")


def normalized_dates(text: str) -> set[str]:
    dates = {
        f"{int(year):04d}-{int(month):02d}-{int(day):02d}"
        for year, month, day in ZH_DATE.findall(text)
    }
    dates.update(
        f"{int(year):04d}-{int(month):02d}-{int(day):02d}"
        for year, month, day in ISO_DATE.findall(text)
    )
    return dates


expected_dates = set()
expected_sources = set()
for payload in compound_baseline:
    record = payload["records"][0]
    for field_name in ("start_date", "end_date"):
        value = record.get(field_name)
        if value:
            expected_dates.update(normalized_dates(value))
    expected_sources.add(payload["citations"][0]["source_name"])

boundary_ambiguity = json.loads(
    lookup_project_schedule.invoke(boundary_examples["ambiguous"])
)["status"] == "ambiguous"
boundary_abstention = json.loads(
    lookup_project_schedule.invoke(boundary_examples["not_found"])
)["status"] == "not_found"

if compound_agent_run["ran"]:
    calls = compound_agent_run["tool_calls"]
    final_answer = compound_agent_run["final_answer"]
    final_dates = normalized_dates(final_answer)
    called_tasks = {
        str(call.get("args", {}).get("task", "")) for call in calls
    }
    expected_tasks = {item["task"] for item in compound_arguments}
    metrics = {
        "tool_selection_accuracy": (
            sum(call.get("name") == "lookup_project_schedule" for call in calls)
            / len(calls)
            if calls
            else 0.0
        ),
        "tool_argument_accuracy": len(called_tasks & expected_tasks)
        / len(expected_tasks),
        "verified_fact_coverage": len(final_dates & expected_dates)
        / len(expected_dates),
        "citation_completeness": sum(
            source in final_answer for source in expected_sources
        )
        / len(expected_sources),
        "invalid_claim_count": len(final_dates - expected_dates),
        "boundary_ambiguity_preservation": boundary_ambiguity,
        "boundary_abstention_preservation": boundary_abstention,
        "llm_calls": compound_agent_run["llm_calls"],
        "tool_calls": len(calls),
        "elapsed_seconds": round(
            compound_agent_run["elapsed_seconds"], 3
        ),
        "input_tokens": compound_agent_run["input_tokens"],
        "output_tokens": compound_agent_run["output_tokens"],
    }
else:
    metrics = {
        "agent_network_run": "skipped",
        "skip_reason": compound_agent_run["skip_reason"],
        "tool_schema_validated": True,
        "agent_constructed": True,
        "deterministic_compound_facts_verified": 3,
        "boundary_ambiguity_preservation": boundary_ambiguity,
        "boundary_abstention_preservation": boundary_abstention,
    }

print(json.dumps(metrics, ensure_ascii=False, indent=2))

{
  "agent_network_run": "skipped",
  "skip_reason": "network run requires RUN_DEEPAGENTS_NETWORK=1",
  "tool_schema_validated": true,
  "agent_constructed": true,
  "deterministic_compound_facts_verified": 3,
  "boundary_ambiguity_preservation": true,
  "boundary_abstention_preservation": true
}


## 9. 工程结论

- **单跳默认直接 pipeline**：一次受控检索和模板回答已经足够；Agent 至少增加模型往返、tool calling 与输出核验，准确日期仍来自相同工具。
- **复合汇总才可能值得**：当用户一次询问多个真实任务、需要逐项失败隔离和统一表达时，模型可负责拆解与组织，但不能接管 Evidence Gate。
- **暂不使用 subagent/filesystem 是工程选择**：当前数据和中间结果都很小，没有上下文隔离或大型文件产物需求。新增能力只会扩大成本和不可预测面。
- **可靠性来源没有改变**：工程路由、词面优先的 hybrid 召回、完整任务记录、字段与引用校验、歧义和拒答。Deep Agents 只是高阶 harness。
- **升级条件**：未来出现多知识源、大结果集、长报告或需并行专业工具时，再用评测决定是否引入 subagents、filesystem 或持久化 memory。

对当前 8 文档场景，推荐路由是：单条事实直接查询；有明确多项拆解需求时才进入这个受限 Deep Agent；任何非 `exact` 状态都回到用户澄清，而不是让模型继续猜。